# Inspect a detector with fresh GPU camera rays

For a **3D camera view illuminated by simulated optical photons**, open
[photon_camera.ipynb](photon_camera.ipynb). The
[optical diagnostics](optical_showcase.ipynb) show sampled paths, spectra and timing.

Each full frame traces **2.5 million camera rays** on the local GPU. These rays
render opaque detector geometry; this is separate from optical photon transport.
Orbit/zoom uses smaller previews and restores full detail after motion settles.
Change **Camera rays** to choose the number traced for each full frame.

Use a CUDA kernel with `torch`, `triton`, `numpy`, `Pillow`, and `ipywidgets`.
The cells below locate this checkout when started from its root or notebook directory.

The main example has about 50,000 20-inch PMTs in a 25.5 m radius water detector.
Use `size=5000., coverage=.05, diameter=304.8` for a faster 326-sensor smoke test.

Select the **TriChroma (GPU, pimm-bench)** kernel registered on this host.

In [ ]:
from pathlib import Path
import sys

root = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "chroma-lite").is_dir() and (p / "chroma-lar").is_dir()), None)
if root is None:
    raise RuntimeError("Start Jupyter from the trichroma checkout")
for source in (root / "chroma-lite", root / "chroma-lar"):
    if str(source) not in sys.path:
        sys.path.insert(0, str(source))

import torch
print(torch.cuda.get_device_name())

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
from chroma_lar.viewer_examples import build_viewer_example

selection = widgets.Dropdown(
    options=[("Theia: 49,684 twenty-inch PMTs", "theia"),
             ("reflect3wires: all six analytic wire planes", "reflect3wires"),
             ("pixelTPC: original averaged pixel faces", "pixelTPC"),
             ("Resolved pixel-pad closeup (32 × 32)", "pixelPads")],
    value="theia", description="Detector:", layout=widgets.Layout(width="440px"))
load_button = widgets.Button(description="Load detector")
output = widgets.Output()

def load_detector(_=None):
    global example, viewer, controls
    if "viewer" in globals():
        viewer.close()
        del viewer
    with output:
        clear_output(wait=True)
        example = build_viewer_example(selection.value)
        print(example.metadata)
        viewer = example.viewer(rays=2_500_000)
        controls = viewer.show()

load_button.on_click(load_detector)
display(widgets.HBox([selection, load_button]), output)
load_detector()

Choose a detector and click **Load detector**. Its existing builder supplies the
geometry; repeated PMTs share GPU meshes. Every selection starts at a useful
interior camera. Orbit, elevation and distance update the view.

- **Theia:** 49,684 20-inch PMTs in illustrative water optics. Use
  `build_viewer_example("theia", radius=5000., coverage=.05, diameter=304.8)`
  for the smaller 326-sensor smoke fixture.
- **reflect3wires:** all six analytic wire planes and 10,750 cylinders, using
  the same FP64 intersection kernel as optical transport. The explicit
  `reflect3wires-mesh` helper retains the slower 32-facet mesh comparison.
  Reduce Distance to inspect individual 0.15 mm wires; overview samples are noisy
  because the wires are smaller than a pixel.
- **pixelTPC:** the original configuration uses area-averaged pixel surfaces.
  Its full-detector view therefore does not resolve the individual pads.
- **Resolved pixel pads:** a separate 32 × 32 pad patch from the original
  pixel-face builder. This is a closeup, not a complete detector.

Opaque rendering does not apply optical transmission. Enclosures, glass,
and pads use triangle geometry. Wires use their validated analytic layer. A
direct `DetectorViewer` call rejects unhandled analytic-wire metadata to avoid
omitting wires.
For a cutaway, pass `hidden_solids=(0,)` to `example.viewer(...)`.

In [ ]:
# A headless frame of the selected detector also works without widgets.
frame = viewer.render_frame()
display(frame)
print(f"{frame.rays:,} camera rays in {frame.seconds*1000:.1f} ms "
      f"({frame.rays_per_second/1e6:.1f}M rays/s)")

Timing includes geometry traversal, shading and RGB download. PNG encoding,
notebook transfer and browser display add overhead. The first frame includes
allocation/JIT. This rate is not a simulated optical-photon rate.

A custom camera can be supplied with
`viewer.render_frame(Camera(eye=(...), target=(...)))`, importing
`Camera` from `chroma.triton.viewer`. Coordinates are in millimetres.

In [ ]:
# When finished with this widget:
# viewer.close()
# del viewer